# Step 01_02_01 -- DuckDB Pre-Ingestion: aoestats

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoestats
**Question:** What does the raw data look like before we commit to an
ingestion strategy? Are there variant column types across weekly files,
type promotions, or NULL patterns we need to handle?
**Invariants applied:** #6 (reproducibility), #9 (step scope)
**Step scope:** query

In [1]:
import logging

import duckdb

from rts_predict.games.aoe2.config import AOESTATS_RAW_DIR
from rts_predict.games.aoe2.datasets.aoestats.pre_ingestion import (
    run_variant_census,
    run_smoke_test,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Pre-ingestion variant census (pyarrow)

In [2]:
census = run_variant_census(AOESTATS_RAW_DIR)
print(f"Matches variant columns: {list(census['matches']['variant_columns'].keys())}")
print(f"Players variant columns: {list(census['players']['variant_columns'].keys())}")

for subdir in ("matches", "players"):
    print(f"\n{subdir}:")
    for col, types in census[subdir]["variant_columns"].items():
        print(f"  {col}: {types}")

INFO:rts_predict.games.aoe2.datasets.aoestats.pre_ingestion:Scanning 172 matches files for variant census
INFO:rts_predict.games.aoe2.datasets.aoestats.pre_ingestion:Scanning 171 players files for variant census


Matches variant columns: ['started_timestamp', 'raw_match_type']
Players variant columns: ['feudal_age_uptime', 'castle_age_uptime', 'imperial_age_uptime', 'profile_id', 'opening']

matches:
  started_timestamp: {'timestamp[us, tz=UTC]': 68, 'timestamp[ns, tz=UTC]': 104}
  raw_match_type: {'double': 66, 'int64': 106}

players:
  feudal_age_uptime: {'double': 82, 'null': 89}
  castle_age_uptime: {'double': 81, 'null': 90}
  imperial_age_uptime: {'double': 81, 'null': 90}
  profile_id: {'double': 36, 'int64': 135}
  opening: {'string': 82, 'null': 89}


## 2. Smoke test

In [3]:
con = duckdb.connect(":memory:")
smoke = run_smoke_test(con, AOESTATS_RAW_DIR)
print(f"Smoke matches: {smoke['matches']['row_count']} rows, {smoke['matches']['column_count']} cols")
print(f"Smoke players: {smoke['players']['row_count']} rows, {smoke['players']['column_count']} cols")

Smoke matches: 371256 rows, 18 cols
Smoke players: 1296181 rows, 14 cols


## 3. DESCRIBE -- column names, types, nullability

In [4]:
for table_name in smoke:
    print(f"\n{'='*60}")
    print(f"  DESCRIBE {table_name}")
    print(f"{'='*60}")
    files = smoke[table_name]["files_sampled"]
    full_paths = [str(AOESTATS_RAW_DIR / table_name / f) for f in files]
    file_list = ", ".join(f"'{p}'" for p in full_paths)
    con.sql(
        f"DESCRIBE SELECT * FROM read_parquet([{file_list}], "
        f"union_by_name=true, filename=true)"
    ).show()


  DESCRIBE matches
┌───────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │       column_type        │  null   │   key   │ default │  extra  │
│      varchar      │         varchar          │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ map               │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ started_timestamp │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ duration          │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ irl_duration      │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ game_id           │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ avg_elo           │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ num_players       │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│

## 4. Row preview -- SELECT * LIMIT 10

In [5]:
for table_name in smoke:
    print(f"\n{'='*60}")
    print(f"  {table_name}: {smoke[table_name]['row_count']} rows, {smoke[table_name]['column_count']} cols")
    print(f"{'='*60}")
    files = smoke[table_name]["files_sampled"]
    full_paths = [str(AOESTATS_RAW_DIR / table_name / f) for f in files]
    file_list = ", ".join(f"'{p}'" for p in full_paths)
    con.sql(
        f"SELECT * FROM read_parquet([{file_list}], "
        f"union_by_name=true, filename=true) LIMIT 10"
    ).show()


  matches: 371256 rows, 18 cols
┌────────────────┬──────────────────────────┬───────────────┬───────────────┬───────────┬────────────────────┬─────────────┬────────────────────┬────────────────────┬─────────────────┬─────────────────┬─────────┬───────┬────────────────┬────────────┬────────────┬──────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      map       │    started_timestamp     │   duration    │ irl_duration  │  game_id  │      avg_elo       │ num_players │     team_0_elo     │     team_1_elo     │ replay_enhanced │   leaderboard   │ mirror  │ patch │ raw_match_type │ game_type  │ game_speed │ starting_age │                                                                        filename                                                                         │
│    varchar     │ timestamp with time zone │     int64     │     int64     │  varchar  │       doubl

## 5. Variant columns -- types and NULL counts per week (matches)

In [6]:
con.sql("""
    SELECT
        filename.split('matches/')[2][:10] AS file_week,
        COUNT(*) AS rows,
        COUNT(raw_match_type) AS raw_match_type_nn,
        COUNT(started_timestamp) AS started_timestamp_nn,
        COUNT(duration) AS duration_nn,
        COUNT(irl_duration) AS irl_duration_nn,
        typeof(duration) AS duration_type,
        typeof(irl_duration) AS irl_duration_type
    FROM read_parquet(
        '{glob}',
        union_by_name=true, filename=true
    )
    GROUP BY file_week
    ORDER BY file_week
""".format(glob=str(AOESTATS_RAW_DIR / "matches" / "*.parquet"))).show()

BinderException: Binder Error: column "duration" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(duration)" if the exact value of "duration" is not important.

## 6. Variant columns -- types and NULL counts per week (players)

In [ ]:
con.sql("""
    SELECT
        filename.split('players/')[2][:10] AS file_week,
        COUNT(*) AS rows,
        COUNT(profile_id) AS profile_id_nn,
        COUNT(feudal_age_uptime) AS feudal_nn,
        COUNT(castle_age_uptime) AS castle_nn,
        COUNT(imperial_age_uptime) AS imperial_nn,
        COUNT(opening) AS opening_nn,
        typeof(profile_id) AS profile_id_type,
        typeof(feudal_age_uptime) AS feudal_type
    FROM read_parquet(
        '{glob}',
        union_by_name=true, filename=true
    )
    GROUP BY file_week
    ORDER BY file_week
""".format(glob=str(AOESTATS_RAW_DIR / "players" / "*.parquet"))).show()

## 7. Ingestion readiness checks

Targeted queries against the full file corpus to resolve open questions
from the smoke test. Each query reads raw files directly (no persistent DB).

### 7a. profile_id precision -- is DOUBLE safe or do we need BIGINT?

In [ ]:
con.sql("""
    SELECT
        MIN(profile_id) AS min_id,
        MAX(profile_id) AS max_id,
        COUNT(*) FILTER (WHERE profile_id IS NULL) AS null_count,
        COUNT(*) FILTER (
            WHERE profile_id IS NOT NULL
              AND profile_id != CAST(CAST(profile_id AS BIGINT) AS DOUBLE)
        ) AS lossy_cast_count,
        COUNT(*) AS total
    FROM read_parquet(
        '{glob}', union_by_name=true
    )
""".format(glob=str(AOESTATS_RAW_DIR / "players" / "*.parquet"))).show()

### 7b. duration / irl_duration value range -- any negatives or extremes?

In [ ]:
con.sql("""
    SELECT
        MIN(duration) AS min_dur_ns,
        MAX(duration) AS max_dur_ns,
        MIN(duration / 1e9) AS min_dur_sec,
        MAX(duration / 1e9) AS max_dur_sec,
        COUNT(*) FILTER (WHERE duration < 0) AS negative_dur,
        COUNT(*) FILTER (WHERE duration IS NULL) AS null_dur,
        MIN(irl_duration) AS min_irl_ns,
        MAX(irl_duration) AS max_irl_ns,
        MIN(irl_duration / 1e9) AS min_irl_sec,
        MAX(irl_duration / 1e9) AS max_irl_sec,
        COUNT(*) FILTER (WHERE irl_duration < 0) AS negative_irl,
        COUNT(*) FILTER (WHERE irl_duration IS NULL) AS null_irl
    FROM read_parquet(
        '{glob}', union_by_name=true
    )
""".format(glob=str(AOESTATS_RAW_DIR / "matches" / "*.parquet"))).show()

### 7c. Join key (game_id) and prediction target (winner) NULL rates

In [ ]:
con.sql("""
    SELECT
        'matches' AS source,
        COUNT(*) AS total,
        COUNT(game_id) AS game_id_nn,
        COUNT(*) - COUNT(game_id) AS game_id_null
    FROM read_parquet('{matches_glob}', union_by_name=true)
    UNION ALL
    SELECT
        'players' AS source,
        COUNT(*) AS total,
        COUNT(game_id) AS game_id_nn,
        COUNT(*) - COUNT(game_id) AS game_id_null
    FROM read_parquet('{players_glob}', union_by_name=true)
""".format(
    matches_glob=str(AOESTATS_RAW_DIR / "matches" / "*.parquet"),
    players_glob=str(AOESTATS_RAW_DIR / "players" / "*.parquet"),
)).show()

In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS total_players,
        COUNT(winner) AS winner_nn,
        COUNT(*) - COUNT(winner) AS winner_null,
        ROUND(100.0 * (COUNT(*) - COUNT(winner)) / COUNT(*), 2) AS winner_null_pct
    FROM read_parquet('{glob}', union_by_name=true)
""".format(glob=str(AOESTATS_RAW_DIR / "players" / "*.parquet"))).show()

### 7d. game_id uniqueness -- duplicates in matches?

In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT game_id) AS distinct_game_ids,
        COUNT(*) - COUNT(DISTINCT game_id) AS duplicate_rows
    FROM read_parquet('{glob}', union_by_name=true)
""".format(glob=str(AOESTATS_RAW_DIR / "matches" / "*.parquet"))).show()

### 7e. overview.json -- DESCRIBE and preview

In [ ]:
con.sql("""
    DESCRIBE SELECT * FROM read_json_auto('{path}')
""".format(path=str(AOESTATS_RAW_DIR / "overview" / "overview.json"))).show()

con.sql("""
    SELECT * FROM read_json_auto('{path}')
""".format(path=str(AOESTATS_RAW_DIR / "overview" / "overview.json"))).show()

## 8. Findings and ingestion strategy recommendation

Summarize all findings from sections 1-7 here after execution.
Key decisions to record:

- **profile_id:** BIGINT or DOUBLE? (based on 7a lossy_cast_count)
- **duration/irl_duration:** keep as BIGINT nanoseconds or convert? (based on 7b range)
- **game_id:** needs dedup at ingestion? (based on 7d)
- **winner NULLs:** acceptable rate? (based on 7c)
- **Proposed DDL** for matches_raw, players_raw, overviews_raw

In [ ]:
con.close()